# Naive beam search on a comic prompt, across seven architectures (Colab)

Written for a Colab GPU runtime, and widened to seven models from an earlier
two-model version that ran on a laptop. This notebook stands alone: it installs
what it needs and imports nothing from the repository it lives in.

The notebook holds a **registry of the models** and walks it in order:
download the weights, load onto the GPU, run the beam search and its greedy
baseline, write the results under `results/<model-slug>/`, free the GPU, delete
the weights from disk, move to the next.

**Why this had to leave the laptop.** The local machine is a 16 GiB CPU-only
Snapdragon X. Measured there, one batched forward pass of Falcon-H1R-7B took
**376 s**, and a warm pass was no faster than a cold one - 14.1 GiB of weights
cannot stay cached in ~8 GiB, so every pass re-streams the model off disk at
~38 MB/s. The full run came to days per model. On a GPU that holds the weights,
the same work is minutes.


<details>
<summary><b>The brief this notebook implements</b> (click to expand)</summary>

```text
Run these models in the cloud. Make a new copy of the notebook under a folder
called cloud.

Make this for all 8 models. The notebook should have a list of the models,
iterate through them, download, load, run experiment, save the results with
an appropriate name, and proceed to the next model, and so on.

Remove the models that don't run.

Add deepseek-ai/DeepSeek-V2-Lite and moonshotai/Kimi-VL-A3B-Thinking.

Look for only language models to keep things simpler. Add Qwen too.
```

The shortlist moved twice. First, two models fitting no single Colab GPU
(Kimi-Linear-48B at 91.5 GiB, DeepSeek-V4-Flash at 294.4 GiB) were replaced by
the largest models from those labs that do fit an A100. Then a first run
revealed that four of the eight were vision-language models, which distorted the
results in ways described in section 2 - so the set was narrowed to text-only
`*ForCausalLM` models, and `Qwen/Qwen3-8B` added back as a dense baseline.

Seven remain. The registry cell records everything dropped and why.
</details>


## 0. Runtime

**Set the runtime first:** *Runtime > Change runtime type > A100 GPU*. Which GPU
you get decides how many of the seven models run at all - the fit check in
section 3 prints the verdict per model before anything downloads.


In [ ]:
# Colab ships torch with CUDA already; only the HF stack needs topping up.
# Pinning nothing on purpose: these architectures are new enough that the
# newest transformers is the one most likely to know about them.
%pip install -q -U transformers accelerate safetensors huggingface_hub

import gc
import json
import math
import os
import re
import shutil
import time
from pathlib import Path

import pandas as pd
import torch
import transformers

SEED = 0
torch.manual_seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

if torch.cuda.is_available():
    DEVICE = "cuda"
    _props = torch.cuda.get_device_properties(0)
    GPU_NAME = _props.name
    VRAM_GIB = _props.total_memory / 2**30
    # bfloat16 needs Ampere or newer. A T4 (Turing) has no bf16 support at all,
    # so fall back to float16 there. On a GPU float16 is fine - the "float16 is
    # slower" rule in this project is a CPU rule and does not apply here.
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DEVICE, GPU_NAME, VRAM_GIB, DTYPE = "cpu", "none", 0.0, torch.float32

print(f"device   {DEVICE}  ({GPU_NAME})")
print(f"VRAM     {VRAM_GIB:.1f} GiB")
print(f"dtype    {DTYPE}")
print(f"torch    {torch.__version__}")
print(f"transformers {transformers.__version__}")

if DEVICE == "cpu":
    print("\nWARNING: no GPU. Runtime > Change runtime type > GPU, then rerun.")


### Where results go

Mounting Drive is what makes this sweep survivable. It runs for an hour or more
and downloads ~173 GiB; a runtime recycled at model six would otherwise take
every finished result with it. With results on Drive, re-running section 6 skips
whatever already completed and carries on.

Set `USE_DRIVE = False` for a throwaway run.


In [ ]:
# Colab wipes /content when the runtime is recycled, and this sweep runs for an
# hour or more across seven models. Results therefore go to Drive, so a
# disconnect costs only the model in flight rather than everything finished so
# far - and section 6 can resume instead of starting over.
#
# Weights deliberately do NOT go to Drive: ~127 GiB in total would exhaust the
# quota, and Drive is far slower to write than local scratch. They stay on the
# local disk and are deleted after each model.
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/beam-search-teddy-bear")
else:
    RESULTS_ROOT = Path("/content/research/results")

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f"results -> {RESULTS_ROOT}")
print("  survives a runtime restart" if USE_DRIVE
      else "  EPHEMERAL - lost if the runtime is recycled")


## 1. Infrastructure, inlined

Locally these helpers live in `src/research/` and the notebooks import them.
There is no `research` package here: the repo has no git remote to install from,
and its `pyproject.toml` pins Python 3.13 and routes `torch` to a **CPU-only**
index - which on a GPU box would silently install a CPU build and quietly defeat
the whole exercise.

So the handful of helpers the experiment needs are reproduced below. `blocks()`
is the same structural search as the local version - the longest `ModuleList`
whose children share a class - not a hard-coded module path, so it survives all
seven naming conventions.


In [ ]:
# --- infrastructure, inlined ------------------------------------------------
# Locally this lives in src/research/ and notebooks import it. There is no
# `research` package on Colab (the repo has no git remote to pip-install from,
# and it pins python 3.13 plus a CPU-only torch index, both wrong here), so the
# few helpers the experiment needs are reproduced below. Behaviour matches the
# local versions; `blocks()` in particular is the same structural search, not a
# hard-coded module path.

ROOT = Path("/content/research")
MODELS_DIR = ROOT / "models"          # HF cache; local scratch, wiped per model
RESULTS = RESULTS_ROOT                # set in the Drive cell above
for _d in (MODELS_DIR, RESULTS):
    _d.mkdir(parents=True, exist_ok=True)

# Point the HF cache at our own folder before huggingface_hub is imported
# anywhere that matters, same intent as research/__init__.py. `os` is imported
# in the environment cell above; what matters here is that this assignment runs
# before the huggingface_hub import on the next line.
os.environ["HF_HUB_CACHE"] = str(MODELS_DIR)

from huggingface_hub import snapshot_download  # noqa: E402
from transformers import AutoTokenizer  # noqa: E402


def model_slug(model_id: str) -> str:
    """google/gemma-4-12B -> google_gemma-4-12B. Filesystem-safe folder name."""
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(model_id).strip()).strip("_")
    if not slug:
        raise ValueError(f"cannot derive a folder name from {model_id!r}")
    return slug


def results_dir(model_id: str) -> Path:
    path = RESULTS / model_slug(model_id)
    path.mkdir(parents=True, exist_ok=True)
    return path


def blocks(model):
    """The repeated decoder blocks, whatever this architecture calls them.

    The longest ModuleList whose entries all share one class. Structural, so it
    works across transformer.h / model.layers / gpt_neox.layers without knowing
    which one this model uses.
    """
    best = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.ModuleList) or len(module) < 2:
            continue
        if len({type(child) for child in module}) != 1:
            continue
        if len(module) > len(best):
            best = [(f"{name}.{i}", child) for i, child in enumerate(module)]
    if not best:
        raise LookupError("could not find a stack of decoder blocks")
    return best


def block_names(model):
    return [name for name, _ in blocks(model)]


def repo_dir(model_id: str) -> Path:
    return MODELS_DIR / f"models--{model_id.replace('/', '--')}"


def disk_free_gib() -> float:
    return shutil.disk_usage("/content").free / 2**30


def download(model_id: str) -> str:
    """Fetch weights, skipping the ONNX/GGUF mirrors some of these repos ship."""
    return snapshot_download(
        model_id,
        cache_dir=MODELS_DIR,
        allow_patterns=["*.safetensors", "*.safetensors.index.json",
                        "*.json", "*.txt", "*.model", "*.jinja"],
        ignore_patterns=["onnx/*", "*.gguf", "*.onnx", "*.pth", "*.bin"],
    )


def auto_classes():
    """Loader classes to try, in order. Not every name exists in every version.

    `AutoModelForCausalLM` is not enough on its own. A model that generates text
    but also accepts images is registered against a vision class instead, and
    asking the wrong Auto class produces "Unrecognized configuration class ...
    for this kind of AutoModel" - which is what skipped Ministral-3-14B on the
    first run, despite it being a perfectly loadable text generator.
    """
    names = ["AutoModelForCausalLM", "AutoModelForImageTextToText",
             "AutoModelForVision2Seq", "AutoModelForSeq2SeqLM"]
    return [(n, getattr(transformers, n)) for n in names if hasattr(transformers, n)]


def load(model_id: str, trust: bool = False):
    """Load onto the GPU at DTYPE. Returns (model, tokenizer).

    `trust` is per model rather than global: only the repos whose registry entry
    says they need custom modelling code get it, so enabling it for DeepSeek
    does not silently enable it for everything else.
    """
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, cache_dir=MODELS_DIR, trust_remote_code=trust)

    tried = []
    for name, cls in auto_classes():
        try:
            model = cls.from_pretrained(
                model_id, cache_dir=MODELS_DIR, dtype=DTYPE,
                device_map=DEVICE, trust_remote_code=trust)
        except ValueError as exc:
            # Only the "wrong Auto class" error is worth trying the next class
            # for. Anything else - out of memory, a missing file, a refused
            # trust_remote_code - is a real failure and must not be swallowed.
            if "Unrecognized configuration class" not in str(exc):
                raise
            tried.append(name)
            continue
        if tried:
            print(f"  (loaded via {name}; {', '.join(tried)} did not accept it)", flush=True)
        model.eval()
        return model, tokenizer

    raise ValueError(
        f"no AutoModel class accepted {model_id}; tried {', '.join(tried)}"
    )


def build_prompt(tokenizer, instruction):
    """Render the instruction the way this model expects it - if it says how.

    Returns ``(text, used_template)``.

    The project rule is never to *assume* a chat template but to ask the
    tokenizer, which is what this does. It matters here: the prompt is an
    instruction, and feeding an instruction to an instruction-tuned model as raw
    completion text is why the first run produced continuations like
    `The user says: "Write a funny story..."` - the model was completing a
    transcript rather than answering.

    Base models have no chat template and correctly fall back to the raw string.
    Which happened is recorded per model, because it changes what the numbers
    mean.
    """
    template = getattr(tokenizer, "chat_template", None)
    if not (USE_CHAT_TEMPLATE and template):
        return instruction, False

    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,      # start the assistant turn, don't just quote the user
    )

    # Many templates embed the BOS token, and the tokenizer will add another one
    # when the search tokenises this string. A doubled BOS is not an error the
    # model will report - it is simply a prompt it was never trained on - so drop
    # the duplicate, but only once it is confirmed the tokenizer really does
    # prepend one.
    bos = getattr(tokenizer, "bos_token", None)
    bos_id = getattr(tokenizer, "bos_token_id", None)
    if bos and bos_id is not None and text.startswith(bos):
        probe = tokenizer("x")["input_ids"]
        if probe[:1] == [bos_id]:
            text = text[len(bos):]

    return text, True


def vocab_size(model):
    """Vocabulary size, wherever this architecture keeps it.

    Multimodal configs nest the text settings under `text_config` and leave
    `vocab_size` unset at the top level - which is why gemma-4-12B reported a
    blank vocabulary on the first run.
    """
    for probe in (model.config, getattr(model.config, "text_config", None)):
        size = getattr(probe, "vocab_size", None) if probe is not None else None
        if size:
            return int(size)
    embeddings = model.get_output_embeddings()
    if embeddings is not None:
        return int(embeddings.weight.shape[0])
    raise LookupError(f"cannot determine vocab_size for {type(model).__name__}")


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def delete_weights(model_id: str):
    """Remove one model's weights from disk.

    Six of these models are 112 GiB of downloads between them, against a Colab
    disk of roughly 110-235 GiB. Unloading from VRAM is not enough - the files
    have to go too, or the run dies on disk part-way through.
    """
    path = repo_dir(model_id)
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)


## 2. The models

**Text-only language models, and that is the point.** An earlier version of this
sweep mixed in four vision-language models, and it went wrong in ways that were
invisible until the results came back:

- `gemma-4-12B` spent its whole 100-token budget emitting `<image|>` placeholder
  tokens. Given no image, the empty image slot dominated its next-token
  distribution - the search was correct, the setup was not.
- `Ministral-3-14B` would not load at all: it registers as
  `Mistral3ForConditionalGeneration`, which `AutoModelForCausalLM` refuses.
- `blocks()` finds the longest stack of identical modules, and on a VLM
  checkpoint that can be the **vision tower** rather than the text decoder - so
  the reported depth may not describe the thing being searched.

Every model below is a `*ForCausalLM` with no vision tower, no image tokens and
no processor config, so none of that can happen. `bf16_GiB` comes from the
safetensors headers on the Hub rather than an estimate.

Six mechanisms across seven models: Mamba-2 at two different SSM-to-attention
ratios, short convolutions, Compressed Convolutional Attention, Multi-head
Latent Attention, and plain dense attention twice - once as a GQA baseline and
once as base weights for the control.

**One mechanism is lost and cannot be recovered here: Gated DeltaNet.** Every
Qwen3.5 checkpoint is a vision model, and Qwen3-Next, which carries GDN, is 80B.
`Qwen/Qwen3-8B` is dense GQA - a baseline, not a substitute.


In [ ]:
# Every model, in the order they run. `bf16_GiB` is measured from the
# safetensors headers on the Hub, not estimated: it is the number the fit check
# below compares against real VRAM.
# Text-only language models, smallest first. Every entry is a `*ForCausalLM`
# with no vision tower, which is what keeps this comparison clean - see the
# markdown above for what the multimodal versions cost.
#
# None needs `trust_remote_code`: transformers recognises all seven config
# classes natively, so no repo's own Python is executed here.
MODELS = [
    {"id": "ibm-granite/granite-4.0-h-tiny",   "params_B":  6.94, "bf16_GiB": 12.9,
     "remote_code": False,
     "family": "Mamba-2 heavy (~9:1 SSM:attention) + MoE"},
    {"id": "tiiuae/Falcon-H1R-7B",             "params_B":  7.59, "bf16_GiB": 14.1,
     "remote_code": False,
     "family": "attention + Mamba-2 SSM interleaved, dense"},
    {"id": "Qwen/Qwen3-8B",                    "params_B":  8.19, "bf16_GiB": 15.3,
     "remote_code": False,
     "family": "dense transformer + grouped-query attention"},
    {"id": "LiquidAI/LFM2.5-8B-A1B",           "params_B":  8.47, "bf16_GiB": 15.8,
     "remote_code": False,
     "family": "short-conv LIV + GQA, sparse MoE (~1.5B active)"},
    {"id": "Zyphra/Zaya1-8B",                  "params_B":  8.84, "bf16_GiB": 16.5,
     "remote_code": False,
     "family": "MoE + Compressed Convolutional Attention (~760M active)"},
    {"id": "mistralai/Mistral-Nemo-Base-2407", "params_B": 12.25, "bf16_GiB": 22.8,
     "remote_code": False,
     "family": "plain dense transformer, base weights - the control"},
    {"id": "deepseek-ai/DeepSeek-V2-Lite",     "params_B": 15.71, "bf16_GiB": 29.3,
     "remote_code": False,
     "family": "Multi-head Latent Attention + DeepSeekMoE (64 experts, 6 active)"},
]

# --- what was dropped, and why -------------------------------------------
#
# Vision-language models. Half the previous shortlist turned out to carry a
# vision tower, which showed up in the results rather than in the plan:
# gemma-4-12B beam-searched its way into emitting <image|> placeholder tokens
# with no image to describe, and Ministral-3 would not load through
# AutoModelForCausalLM at all because it registers as ForConditionalGeneration.
#
#   Qwen/Qwen3.5-9B                    Qwen3_5ForConditionalGeneration
#   google/gemma-4-12B                 Gemma4UnifiedForConditionalGeneration
#   mistralai/Ministral-3-14B-...      Mistral3ForConditionalGeneration
#   moonshotai/Kimi-VL-A3B-Thinking    KimiVLForConditionalGeneration
#
# Too large for one A100, at any precision. Still analysable from safetensors
# headers without downloading - see scripts/compare_architectures.py:
#
#   moonshotai/Kimi-Linear-48B-A3B-Base   49.12 B    91.5 GiB
#   deepseek-ai/DeepSeek-V4-Flash        158.07 B   294.4 GiB
#
# One mechanism is lost by going text-only and cannot be recovered in this size
# band: **Gated DeltaNet**. Every Qwen3.5 checkpoint (9B, 4B, 2B) is a vision
# model, and Qwen3-Next - which does carry GDN - is 80B. Qwen/Qwen3-8B above is
# dense GQA, so it is a strong baseline rather than a replacement mechanism.

pd.DataFrame(MODELS)[["id", "params_B", "bf16_GiB", "family"]]


## 3. Configuration, and what will actually fit

`CONFIG` is the search itself: width 10, 100 tokens, top-10 per beam, seed 0.

The fit check still matters even though the models that never fit have been
removed, because the seven do not all fit every Colab tier. Weights must sit in
VRAM with room left for activations and the logits tensor, so the budget is
total VRAM minus a headroom allowance:

| Runtime | VRAM | How many of the seven run |
|---|---|---|
| T4 | 15 GB | none reliably - and no bfloat16, so it falls back to float16 |
| L4 | 22.5 GB | the first five (up to Zaya1-8B) |
| A100 | 40 GB | **all seven** |

**An A100 runtime is what this notebook wants.** DeepSeek-V2-Lite at 29.3 GiB is
the ceiling and exists only on that tier.

Nothing is downloaded until section 6, so read the verdict below before starting
a run: a wrong tier costs you a 13-29 GiB download per model before it fails.


In [ ]:
PROMPT = "Write a funny story about a teddy bear in about 100 tokens."

# Ask each tokenizer whether it has a chat template, and use it when it does.
#
# The first run of this sweep did not, and every instruction-tuned model treated
# the prompt as text to continue rather than an instruction to follow - hence
# outputs like `The user says: "Write a funny story about a teddy bear..."`
# repeated until the token budget ran out.
#
# Set False to reproduce that raw-completion behaviour, which is the right
# setting if the question is about the *base* next-token distribution rather
# than about instruction following. Base models have no template either way and
# are unaffected; `used_chat_template` in each summary row records what actually
# happened per model.
USE_CHAT_TEMPLATE = True

CONFIG = dict(
    num_tokens_to_generate=100,  # how many new tokens to generate
    beam_width=10,               # hypotheses kept alive after every step
    num_return_sequences=4,      # how many of them to return at the end
    top_k_per_beam=10,           # next-token candidates per beam (None = whole vocabulary)
    temperature=1.0,             # applied to the logits before the softmax
    early_stopping=True,         # stop once every surviving beam has hit EOS
    record_trace=True,           # analysis mode; False is the low-memory mode
    device=None,                 # None = follow the model. It is on the GPU.
    use_kv_cache=True,           # see below - the laptop notebooks run without it
)

# The laptop notebooks run with use_kv_cache=False: recomputing the whole
# sequence every step trades compute for memory, which is the right trade on a
# 16 GiB CPU box. On an A100 that trade is backwards, and it is expensive -
# without a cache the cost is O(n^2) in generated length, so 100 tokens is ~40x
# the work of 10 rather than 10x.
#
# Correctness is the thing to be careful about, not speed. Beam search re-parents
# its hypotheses every step, so the cache has to be permuted to match or one
# beam's history gets applied to another beam's tokens - fluent, plausible,
# wrong. `reorder_cache()` does that permutation, and
# `tests/test_beam.py::test_kv_cache_matches_uncached` pins the cached path to
# the uncached one on a model whose every step is checkable by hand.
#
# Set it False if a model's cache cannot be reordered - reorder_cache() raises
# a TypeError naming the type rather than guessing.

# No model in the current registry needs this: transformers recognises all seven
# config classes natively, so nothing here executes Python fetched from the Hub.
# The switch is kept because it is per model, not global - if a repo is added
# later whose entry sets `remote_code: True`, only that repo receives it, and
# only once this is deliberately turned on.
TRUST_REMOTE_CODE = False


def out_name(name: str) -> str:
    """Output filename for the current run, tagged when the KV cache is on.

    A cached and an uncached run of the same model write into the same
    `results/<model-slug>/` folder, so without this the second silently
    overwrites the first. Tagging the cached run keeps both, which is what makes
    them comparable - and `use_kv_cache` is recorded in run-metadata.json either
    way, so the filename is a convenience rather than the source of truth.
    """
    return f"kvcached-{name}" if CONFIG.get("use_kv_cache") else name

# Headroom for activations, the logits tensor and allocator fragmentation.
# Weights are only part of what has to fit.
VRAM_HEADROOM_GIB = 2.5
VRAM_BUDGET_GIB = max(VRAM_GIB - VRAM_HEADROOM_GIB, 0.0)

print(f"VRAM budget for weights: {VRAM_BUDGET_GIB:.1f} GiB "
      f"({VRAM_GIB:.1f} total - {VRAM_HEADROOM_GIB} headroom)")
print(f"disk free: {disk_free_gib():.0f} GiB\n")

runnable = 0
for m in MODELS:
    if m["bf16_GiB"] > VRAM_BUDGET_GIB:
        verdict = f"SKIP - needs {m['bf16_GiB']:.1f} GiB"
    elif m.get("remote_code") and not TRUST_REMOTE_CODE:
        verdict = "SKIP - needs TRUST_REMOTE_CODE"
    else:
        verdict = "runs"
        runnable += 1
    print(f"  {m['id']:45s} {m['bf16_GiB']:6.1f} GiB   {verdict}")

total_download = sum(m["bf16_GiB"] for m in MODELS
                     if m["bf16_GiB"] <= VRAM_BUDGET_GIB
                     and (TRUST_REMOTE_CODE or not m.get("remote_code")))
print(f"\n{runnable} of {len(MODELS)} will run, ~{total_download:.0f} GiB to download "
      f"in total (peak disk is one model at a time)")


## 4. The implementation

Copied verbatim from the local notebooks, not imported, so this notebook stands
on its own. `tests/test_beam.py` executes the tagged cells of every notebook
that carries them, so this copy is tested too and cannot drift.

It needed **no changes for the GPU**: `beam_search(..., device=None)` resolves
the device from the model's own parameters, so putting the model on CUDA is
enough.


### 4a. Implementation, part 1 of 3

In [ ]:
# Imported here as well as in the setup cell, so the three implementation cells
# below stand on their own if lifted out of this notebook.
import math
from dataclasses import dataclass, field
from inspect import Parameter, signature

import torch

ROOT_BEAM_ID = 0        # the prompt itself, before any token is generated


@dataclass
class BeamSearchResult:
    """What the search produced, as plain Python - hand it to pandas or print it."""

    prompt: str
    prompt_token_ids: list
    config: dict
    sequences: list = field(default_factory=list)   # ranked, num_return_sequences of them
    trace: list = field(default_factory=list)       # one row per selected beam per step
    steps_run: int = 0
    stopped_early: bool = False


def validate_config(num_tokens_to_generate, beam_width, num_return_sequences,
                    top_k_per_beam, temperature):
    """Reject configurations that cannot mean what they say, before any compute."""
    if num_tokens_to_generate < 1:
        raise ValueError("num_tokens_to_generate must be at least 1")
    if beam_width < 1:
        raise ValueError("beam_width must be at least 1")
    if not 1 <= num_return_sequences <= beam_width:
        raise ValueError(
            f"num_return_sequences={num_return_sequences} must be between 1 and "
            f"beam_width={beam_width}: only beam_width hypotheses survive"
        )
    if top_k_per_beam is not None and top_k_per_beam < beam_width:
        raise ValueError(
            f"top_k_per_beam={top_k_per_beam} is below beam_width={beam_width}. The "
            "first step expands the prompt alone, so it could not fill the beam. "
            "Pass None to use the whole vocabulary."
        )
    if not temperature > 0:
        raise ValueError("temperature must be positive")


def eos_token_ids(model, tokenizer):
    """Every id that ends a sequence, asking the model first and the tokenizer second.

    Some models declare several (an EOS plus a chat end-of-turn token); some
    tokenizers declare none at all. Both are handled rather than assumed.
    """
    raw = getattr(getattr(model, "generation_config", None), "eos_token_id", None)
    if raw is None:
        raw = getattr(tokenizer, "eos_token_id", None)
    if raw is None:
        return set()
    if isinstance(raw, int):
        return {raw}
    return {int(token_id) for token_id in raw if token_id is not None}


def resolve_device(model, requested):
    """Where to build the id tensors: the model's own device, unless told otherwise."""
    model_device = next(model.parameters()).device
    if requested is None:
        return model_device
    device = torch.device(requested)
    if device.type != model_device.type:
        raise ValueError(
            f"device={requested!r} but the model is on {model_device}. "
            "Move the model first; beam_search does not relocate it."
        )
    return model_device


def logits_to_keep_arg(model):
    """Name of the forward argument that trims logits to the last position, if any.

    Recent transformers exposes `logits_to_keep` (older: `num_logits_to_keep`) on
    causal LMs. Passing 1 makes the model run the LM head for the final position
    only, turning a (beams, length, vocab) tensor into (beams, 1, vocab) - the
    largest single allocation in the loop. On an architecture without it, the
    last position is sliced out instead.
    """
    parameters = signature(model.forward).parameters
    for name in ("logits_to_keep", "num_logits_to_keep"):
        parameter = parameters.get(name)
        if parameter is not None and parameter.kind is not Parameter.VAR_KEYWORD:
            return name
    return None


def reorder_cache(model, past, index):
    """Reindex a cache's batch dimension so row i continues beam `index[i]`.

    Beam search re-parents its hypotheses every step: the beam in row 3 after a
    step may descend from the beam that was in row 7 before it. A cache is
    indexed by batch row, so it has to be permuted to match or every subsequent
    forward pass applies one beam's history to another beam's tokens - which
    produces fluent, plausible, wrong output rather than an error.

    Three routes, most authoritative first. The model's own `_reorder_cache` is
    preferred because a cache is not always keys and values: the Mamba-2 layers
    in Falcon-H1R and the short convolutions in LFM2.5 carry recurrent conv and
    SSM state, which only the architecture knows how to permute.
    """
    if past is None:
        return None

    own = getattr(model, "_reorder_cache", None)
    if callable(own):
        return own(past, index)

    select = getattr(past, "batch_select_indices", None)
    if callable(select):
        select(index)               # modern Cache objects mutate in place
        return past

    if isinstance(past, tuple):     # legacy tuple-of-(key, value)-tuples
        return tuple(
            tuple(tensor.index_select(0, index) for tensor in layer)
            for layer in past
        )

    raise TypeError(
        f"cannot reorder a cache of type {type(past).__name__} for beam search; "
        "rerun with use_kv_cache=False"
    )


def next_token_log_probs(model, sequences, temperature, logits_arg,
                         past=None, use_kv_cache=False):
    """Log probabilities of the next token for every beam: (beams, vocab).

    Returns ``(log_probs, past)``.

    With ``use_kv_cache=False`` no cache is passed in and none is kept: the model
    recomputes from the whole sequence every step. That is the memory-efficient
    constraint the laptop notebooks are built to, and it costs O(n^2) in
    generated length.

    With ``use_kv_cache=True`` only the newest token is fed and the rest comes
    from ``past``. The attention mask still spans the *full* sequence - past plus
    new - which is what tells the model how much history the cache holds.
    """
    kwargs = {"use_cache": bool(use_kv_cache)}
    if logits_arg is not None:
        kwargs[logits_arg] = 1

    if use_kv_cache and past is not None:
        inputs = sequences[:, -1:]          # everything before it is in `past`
        kwargs["past_key_values"] = past
    else:
        inputs = sequences

    outputs = model(
        input_ids=inputs,
        attention_mask=torch.ones_like(sequences),
        **kwargs,
    )
    # .clone() matters: `[:, -1, :]` is a view onto the full logits buffer, so
    # without it `del outputs` would free nothing at all.
    logits = outputs.logits[:, -1, :].to(torch.float32).clone()
    new_past = getattr(outputs, "past_key_values", None) if use_kv_cache else None
    del outputs

    if temperature != 1.0:
        logits /= temperature
    log_probs = torch.log_softmax(logits, dim=-1)
    del logits
    return log_probs, new_past

### 4b. Implementation, part 2 of 3

In [ ]:
def final_row(*, rank, beam_id, ids, log_probs, total, is_finished, prompt, tokenizer):
    """One ranked output sequence, with the aggregates that describe its score."""
    count = len(log_probs)
    mean_log_prob = total / count if count else float("nan")
    text = tokenizer.decode(ids)
    return {
        "rank": rank,
        "beam_id": beam_id,
        "generated_text": text,
        "full_text": prompt + text,
        "generated_token_ids": ids,
        "generated_tokens": [tokenizer.decode([token_id]) for token_id in ids],
        "token_log_probabilities": log_probs,
        "num_generated_tokens": count,
        "total_log_probability": total,
        "sequence_probability": math.exp(total),
        "average_token_log_probability": mean_log_prob,
        # Geometric mean, i.e. exp(mean log p): the per-token probability that
        # reproduces the sequence probability when multiplied out. Deliberately
        # not the arithmetic mean of the token probabilities.
        "average_token_probability": math.exp(mean_log_prob) if count else float("nan"),
        "finished": is_finished,
    }


def annotate_survival(trace, *, surviving, returned):
    """Mark which traced beams lived on, and which are ancestors of a returned one.

    A beam that is neither the parent of a later row nor alive at the end was
    discarded at the step where it appears.
    """
    parents = {row["parent_beam_id"] for row in trace}
    alive = set(surviving)

    parent_of = {row["beam_id"]: row["parent_beam_id"] for row in trace}
    on_path = set()
    for beam_id in returned:
        while beam_id != ROOT_BEAM_ID and beam_id not in on_path:
            on_path.add(beam_id)
            beam_id = parent_of.get(beam_id, ROOT_BEAM_ID)

    for row in trace:
        row["survived"] = row["beam_id"] in parents or row["beam_id"] in alive
        row["on_final_path"] = row["beam_id"] in on_path


def token_level_rows(result):
    """One row per token of every returned sequence, with its running score.

    Answers "how did this sequence get its probability?" - which token was cheap,
    and which one the model had to be talked into.
    """
    rows = []
    for sequence in result.sequences:
        cumulative = 0.0
        tokens = sequence["generated_tokens"]
        log_probs = sequence["token_log_probabilities"]
        for position, token_id in enumerate(sequence["generated_token_ids"]):
            cumulative += log_probs[position]
            rows.append(
                {
                    "rank": sequence["rank"],
                    "beam_id": sequence["beam_id"],
                    "position": position + 1,
                    "token_id": token_id,
                    "token": tokens[position],
                    "token_probability": math.exp(log_probs[position]),
                    "token_log_probability": log_probs[position],
                    "cumulative_log_probability": cumulative,
                    "cumulative_probability": math.exp(cumulative),
                }
            )
    return rows

### 4c. Implementation, part 3 of 3

In [ ]:
@torch.inference_mode()             # no autograd graph, no gradient buffers
def beam_search(model, tokenizer, prompt, *, num_tokens_to_generate=10, beam_width=10,
                num_return_sequences=4, top_k_per_beam=10, temperature=1.0,
                early_stopping=True, record_trace=True, device=None,
                use_kv_cache=False):
    """Generate tokens by beam search and return the highest-probability sequences."""
    validate_config(num_tokens_to_generate, beam_width, num_return_sequences,
                    top_k_per_beam, temperature)
    config = dict(num_tokens_to_generate=num_tokens_to_generate, beam_width=beam_width,
                  num_return_sequences=num_return_sequences, top_k_per_beam=top_k_per_beam,
                  temperature=temperature, early_stopping=early_stopping,
                  record_trace=record_trace, device=device,
                  use_kv_cache=use_kv_cache)

    model.eval()          # a live dropout would silently corrupt every probability here
    device = resolve_device(model, device)
    logits_arg = logits_to_keep_arg(model)
    eos_ids = eos_token_ids(model, tokenizer)
    eos_tensor = torch.tensor(sorted(eos_ids), device=device, dtype=torch.long)
    # Only ever appended to a beam that has already finished, which cannot happen
    # unless the model declares an EOS token.
    filler_id = min(eos_ids) if eos_ids else 0

    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    if prompt_ids.shape[1] == 0:
        raise ValueError("the prompt tokenised to nothing; beam search needs a context")

    # The entire live state of the search. `sequences` starts as the prompt alone:
    # all beams would otherwise be identical at step 1.
    sequences = prompt_ids
    scores = torch.zeros(1, device=device, dtype=torch.float32)
    finished = torch.zeros(1, device=device, dtype=torch.bool)
    beam_ids = [ROOT_BEAM_ID]
    generated_ids = [[]]
    token_log_probs = [[]]

    next_beam_id = ROOT_BEAM_ID + 1
    trace, steps_run, stopped_early = [], 0, False
    past = None                     # only ever populated when use_kv_cache

    for step in range(1, num_tokens_to_generate + 1):
        log_probs, past = next_token_log_probs(
            model, sequences, temperature, logits_arg, past, use_kv_cache)
        vocab_size = log_probs.shape[-1]
        top_k = min(top_k_per_beam or vocab_size, vocab_size)
        candidate_log_probs, candidate_ids = torch.topk(log_probs, top_k, dim=-1)
        del log_probs           # the (beams, vocab) tensor dies here; it is never stored

        # A finished beam carries forward on one filler token of log probability 0,
        # so its score is untouched and every beam keeps the same length.
        if bool(finished.any()):
            candidate_log_probs[finished] = -math.inf
            candidate_log_probs[finished, 0] = 0.0
            candidate_ids[finished, 0] = filler_id

        # Every candidate from every beam, scored on one axis, then compared globally.
        candidate_scores = scores.unsqueeze(1) + candidate_log_probs
        flat = candidate_scores.reshape(-1)
        best_scores, best_flat = torch.topk(flat, min(beam_width, flat.numel()))
        del candidate_scores, flat

        finite = torch.isfinite(best_scores)
        if not bool(finite.all()):      # only reachable if the model rules out most tokens
            best_scores, best_flat = best_scores[finite], best_flat[finite]
        if best_scores.numel() == 0:
            raise RuntimeError(f"step {step}: no beam has a finite continuation")
        keep = best_scores.numel()

        parents = torch.div(best_flat, top_k, rounding_mode="floor")
        slots = best_flat % top_k
        chosen_ids = candidate_ids[parents, slots]
        chosen_log_probs = candidate_log_probs[parents, slots]
        previous_scores = scores[parents]
        was_finished = finished[parents]
        del candidate_ids, candidate_log_probs, best_flat, slots, finite

        parent_list = parents.tolist()
        chosen_id_list = chosen_ids.tolist()
        chosen_log_prob_list = chosen_log_probs.tolist()
        carried_list = was_finished.tolist()
        new_beam_ids = list(range(next_beam_id, next_beam_id + keep))
        next_beam_id += keep

        new_generated, new_token_log_probs = [], []
        for parent, token_id, token_log_prob, carried in zip(
            parent_list, chosen_id_list, chosen_log_prob_list, carried_list, strict=True
        ):
            # The filler token of a carried-forward beam was never generated, so it
            # belongs neither in that beam's tokens nor in its score history.
            new_generated.append(
                list(generated_ids[parent]) if carried
                else [*generated_ids[parent], token_id]
            )
            new_token_log_probs.append(
                list(token_log_probs[parent]) if carried
                else [*token_log_probs[parent], token_log_prob]
            )

        if record_trace:
            rows = zip(parent_list, new_beam_ids, chosen_id_list, chosen_log_prob_list,
                       previous_scores.tolist(), best_scores.tolist(), carried_list,
                       strict=True)
            for rank, (parent, bid, token_id, log_prob, previous, score, carried) in enumerate(
                rows
            ):
                trace.append({
                    "step": step,
                    "parent_beam_id": beam_ids[parent],
                    "beam_id": bid,
                    "beam_rank": rank,              # 0 is the best-scoring beam of this step
                    "token_id": token_id,
                    "token": tokenizer.decode([token_id]),
                    "token_probability": math.exp(log_prob),
                    "token_log_probability": log_prob,
                    "previous_log_probability": previous,
                    "cumulative_log_probability": score,
                    "is_eos": token_id in eos_ids,
                    # True when the beam had already finished and this row only
                    # holds its place - not a real generation event.
                    "carried_forward": carried,
                })

        sequences = torch.cat([sequences[parents], chosen_ids.unsqueeze(1)], dim=1)
        if use_kv_cache:
            # Must happen with the same `parents` used to reorder `sequences`
            # above, and before it is deleted below.
            past = reorder_cache(model, past, parents)
        scores = best_scores
        finished = was_finished | torch.isin(chosen_ids, eos_tensor)
        beam_ids = new_beam_ids
        generated_ids = new_generated
        token_log_probs = new_token_log_probs
        del parents, chosen_ids, chosen_log_probs, previous_scores, was_finished

        steps_run = step
        if early_stopping and bool(finished.all()):
            stopped_early = True
            break

    order = torch.argsort(scores, descending=True).tolist()
    returned = order[:num_return_sequences]

    result = BeamSearchResult(prompt=prompt, prompt_token_ids=prompt_ids[0].tolist(),
                              config=config, steps_run=steps_run, stopped_early=stopped_early)
    result.sequences = [
        final_row(rank=rank, beam_id=beam_ids[b], ids=generated_ids[b],
                  log_probs=token_log_probs[b], total=float(scores[b]),
                  is_finished=bool(finished[b]), prompt=prompt, tokenizer=tokenizer)
        for rank, b in enumerate(returned, start=1)
    ]
    if record_trace:
        annotate_survival(trace, surviving=[beam_ids[b] for b in order],
                          returned=[beam_ids[b] for b in returned])
    result.trace = trace
    return result

## 5. Verify the KV cache before spending anything

`CONFIG` sets `use_kv_cache=True`, which is a large saving and also the one
change here that can be wrong **silently**. Beam search re-parents its
hypotheses every step, so the cache must be permuted to match; if it is not,
every later forward pass applies one beam's history to another beam's tokens and
the search returns fluent, plausible, wrong text. No exception, no warning.

That is worth catching before ~127 GiB of downloads rather than after, so the
checks run here, on a four-token toy model whose every step can be worked out by
hand. Three things are checked, and the second and third matter as much as the
first:

1. **The cached search returns exactly what the uncached search returns.** The
   uncached path is the one the repo's `tests/test_beam.py` pins against brute
   force, so matching it inherits that guarantee.
2. **The cache is actually used** - one token per pass after the prompt. Without
   this, an implementation that accepts the cache and then ignores it would pass
   check 1 while saving nothing at all.
3. **The cache is reordered to follow the beams** - each row's cached history is
   compared against that beam's real lineage, walked back through
   `parent_beam_id`. This is the check that fails if `reorder_cache` is dropped.

A failure raises rather than warns: bad results are worse than no results.


In [ ]:
# Self-contained check of the KV-cache path, run before any weights download.
# The repo's tests/test_beam.py is not here, so the toy model comes with it - and
# it exercises the beam_search defined in the cells above, not a copy of it.
from types import SimpleNamespace  # noqa: E402

from torch import nn  # noqa: E402

_VOCAB = "abcd"
_PROBS = torch.tensor([                 # row i is P(next | last token was i)
    [0.05, 0.80, 0.10, 0.05],
    [0.10, 0.10, 0.20, 0.60],
    [0.05, 0.75, 0.15, 0.05],
    [0.42, 0.30, 0.20, 0.08],
])


class _ToyTokenizer:
    eos_token_id = None

    def __call__(self, text, return_tensors=None):
        return {"input_ids": torch.tensor([[_VOCAB.index(c) for c in text]])}

    def decode(self, ids):
        return "".join(_VOCAB[int(i)] for i in ids)


class _ToyCachingModel(nn.Module):
    """Markov toy whose "cache" is the token history itself.

    A real cache holds keys and values; holding the ids instead is what makes a
    mis-ordering *visible* - after every step each row's cache must contain that
    row's own tokens, so a wrong permutation is a mismatch rather than merely
    different-looking text.
    """

    def __init__(self):
        super().__init__()
        self.marker = nn.Parameter(torch.zeros(1))
        self.log_probs = _PROBS.log()
        self.generation_config = SimpleNamespace(eos_token_id=None)
        self.calls, self.seen = [], []

    def forward(self, input_ids, attention_mask=None, logits_to_keep=0,
                past_key_values=None, use_cache=False, **kwargs):
        full = input_ids if past_key_values is None else torch.cat(
            [past_key_values, input_ids], dim=1)
        self.calls.append(input_ids.shape[1])
        self.seen.append(full.clone())
        wanted = full[:, -logits_to_keep:] if logits_to_keep else full
        return SimpleNamespace(logits=self.log_probs[wanted],
                               past_key_values=full if use_cache else None)

    def _reorder_cache(self, past, index):
        return past.index_select(0, index)


# A multi-character prompt on purpose: _ToyTokenizer is character-level, so a
# one-character prompt makes "the first pass fed the whole prompt" true of any
# implementation, cached or not, and check 2 below could never fail. The toy
# logits depend only on the last token, so a longer prompt changes nothing else.
_PROMPT = "abc"
_ARGS = dict(num_tokens_to_generate=4, beam_width=3, top_k_per_beam=3,
             num_return_sequences=3)
_failures = []

# 1. the cached search must return exactly what the uncached search returns
_plain = beam_search(_ToyCachingModel(), _ToyTokenizer(), _PROMPT, use_kv_cache=False, **_ARGS)
_cached = beam_search(_ToyCachingModel(), _ToyTokenizer(), _PROMPT, use_kv_cache=True, **_ARGS)
if [s["generated_token_ids"] for s in _cached.sequences] != \
   [s["generated_token_ids"] for s in _plain.sequences]:
    _failures.append("cached and uncached searches returned different sequences")
for _got, _want in zip(_cached.sequences, _plain.sequences, strict=True):
    if abs(_got["total_log_probability"] - _want["total_log_probability"]) > 1e-6:
        _failures.append("cached and uncached scores differ")
        break

# 2. the cache must actually be used - otherwise (1) passes while saving nothing
_m = _ToyCachingModel()
beam_search(_m, _ToyTokenizer(), _PROMPT, use_kv_cache=True, **_ARGS)
if _m.calls[0] != len(_PROMPT) or set(_m.calls[1:]) != {1}:
    _failures.append(
        f"expected {len(_PROMPT)} prompt tokens then one per pass, got {_m.calls}")

# 3. after re-parenting, each row's cache must hold that row's own history.
#    This is the check that fails if reorder_cache were dropped.
_m = _ToyCachingModel()
_res = beam_search(_m, _ToyTokenizer(), _PROMPT, use_kv_cache=True, record_trace=True, **_ARGS)
_parent = {r["beam_id"]: r["parent_beam_id"] for r in _res.trace}
_token = {r["beam_id"]: r["token_id"] for r in _res.trace}
_last = max(r["step"] for r in _res.trace)
for _rank, _row in enumerate(r for r in _res.trace if r["step"] == _last):
    _lineage, _node = [], _row["beam_id"]
    while _node in _token:
        _lineage.append(_token[_node])
        _node = _parent[_node]
    _lineage.reverse()
    # seen[-1] is the input to the *last* forward pass, which ran before this
    # beam's final token existed - so it holds the lineage minus that token.
    # Comparing against the full lineage is off by one and fails on a correct
    # implementation. Slice from the left so an empty history stays honest.
    _history = _lineage[:-1]
    _shown = _m.seen[-1][_rank].tolist()
    if _shown[len(_shown) - len(_history):] != _history:
        _failures.append(f"row {_rank}: cache holds {_shown}, beam history ends {_history}")
        break

if _failures:
    for _f in _failures:
        print("FAIL:", _f)
    if CONFIG.get("use_kv_cache"):
        raise AssertionError(
            "the KV-cache path is wrong; results would be silently incorrect. "
            "Set use_kv_cache=False in CONFIG to run without it."
        )
else:
    print("KV cache verified: matches the uncached search, is actually used, "
          "and is reordered to follow the beams.")


## 6. One model, end to end

Download, load, search, save, free, delete. The saves happen **before** anything
is released, so a failure on a later model cannot cost the results of an earlier
one.

Weight deletion is not housekeeping - it is load-bearing. The seven models are
**~127 GiB of downloads** between them, against a Colab disk of roughly
110-235 GiB - so on a smaller disk the sweep cannot hold them all at once.
Deleting each one keeps peak disk at a single model (~29 GiB at worst), and it
happens in a `finally`, including after a failed partial download.


In [ ]:
def run_one(spec):
    """Download, load, search, save, free, delete. Returns one summary row."""
    model_id = spec["id"]
    out = results_dir(model_id)

    if spec["bf16_GiB"] > VRAM_BUDGET_GIB:
        raise MemoryError(
            f"needs {spec['bf16_GiB']:.1f} GiB of weights but the budget is "
            f"{VRAM_BUDGET_GIB:.1f} GiB on this {GPU_NAME}"
        )

    needs_code = spec.get("remote_code", False)
    if needs_code and not TRUST_REMOTE_CODE:
        raise PermissionError(
            f"{model_id} ships its own modelling code and cannot load without "
            f"trust_remote_code; set TRUST_REMOTE_CODE = True in section 3 if "
            f"you have decided to trust this repo"
        )

    print(f"  disk free {disk_free_gib():.0f} GiB - downloading", flush=True)
    t0 = time.perf_counter()
    download(model_id)
    download_seconds = time.perf_counter() - t0
    print(f"  downloaded in {download_seconds:.0f}s", flush=True)

    t0 = time.perf_counter()
    model, tokenizer = load(model_id, trust=needs_code)
    load_seconds = time.perf_counter() - t0
    vram = torch.cuda.memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    print(f"  loaded in {load_seconds:.0f}s, {vram:.1f} GiB on GPU", flush=True)

    try:
        t0 = time.perf_counter()
        prompt, used_template = build_prompt(tokenizer, PROMPT)
        print(f"  chat template: {'applied' if used_template else 'none - raw completion'}",
              flush=True)

        t0 = time.perf_counter()
        result = beam_search(model, tokenizer, prompt, **CONFIG)
        search_seconds = time.perf_counter() - t0
        print(f"  beam search {search_seconds:.0f}s", flush=True)

        t0 = time.perf_counter()
        greedy = beam_search(model, tokenizer, prompt, **{
            **CONFIG, "beam_width": 1, "top_k_per_beam": 1, "num_return_sequences": 1,
        })
        print(f"  greedy {time.perf_counter() - t0:.0f}s", flush=True)

        best, greedy_best = result.sequences[0], greedy.sequences[0]
        trace_df = pd.DataFrame(result.trace)

        # --- save, named for the model, before anything is freed -------------
        final_csv = pd.DataFrame(result.sequences)
        for column, join in (("generated_token_ids", " "), ("generated_tokens", "|")):
            final_csv[column] = final_csv[column].apply(
                lambda values, sep=join: sep.join(str(v) for v in values))
        final_csv["token_log_probabilities"] = final_csv["token_log_probabilities"].apply(
            lambda values: " ".join(f"{v:.6f}" for v in values))
        final_csv.to_csv(out / out_name("final_results.csv"), index=False)
        trace_df.to_csv(out / out_name("beam_trace.csv"), index=False)
        pd.DataFrame(token_level_rows(result)).to_csv(
            out / out_name("token-level-probabilities.csv"), index=False)
        pd.DataFrame(token_level_rows(greedy)).to_csv(
            out / out_name("greedy_baseline.csv"), index=False)
        (out / out_name("run-metadata.json")).write_text(json.dumps({
            "model": model_id, "family": spec["family"],
            "instruction": PROMPT,          # what was asked
            "prompt": prompt,               # what the model was actually given
            "used_chat_template": used_template,
            "dtype": str(DTYPE), "device": DEVICE, "gpu": GPU_NAME, "seed": SEED,
            "config": {k: v for k, v in CONFIG.items()},
            "download_s": round(download_seconds, 1),
            "load_s": round(load_seconds, 1),
            "search_s": round(search_seconds, 1),
        }, indent=2))

        token_lists = [tuple(s["generated_token_ids"]) for s in result.sequences]

        def shared_prefix(first, second):
            length = 0
            for left, right in zip(first, second, strict=False):
                if left != right:
                    break
                length += 1
            return length

        pairs = [shared_prefix(a, b)
                 for i, a in enumerate(token_lists) for b in token_lists[i + 1:]]

        row = {
            "model": model_id,
            "family": spec["family"],
            "params_B": spec["params_B"],
            "bf16_GiB": spec["bf16_GiB"],
            "used_chat_template": used_template,
            "blocks": len(block_names(model)),
            "vocab": vocab_size(model),
            "gpu_GiB": round(vram, 2),
            "download_s": round(download_seconds, 1),
            "load_s": round(load_seconds, 1),
            "search_s": round(search_seconds, 1),
            "beam_text": best["generated_text"],
            "beam_log_p": best["total_log_probability"],
            "beam_avg_token_p": best["average_token_probability"],
            "greedy_text": greedy_best["generated_text"],
            "greedy_log_p": greedy_best["total_log_probability"],
            "greedy_avg_token_p": greedy_best["average_token_probability"],
            "gain_log_p": best["total_log_probability"] - greedy_best["total_log_probability"],
            "mean_pairwise_prefix": round(sum(pairs) / len(pairs), 2) if pairs else 0.0,
            "distinct_first_tokens": len({t[0] for t in token_lists if t}),
            "discarded": int((~trace_df["survived"]).sum()),
            "steps_run": result.steps_run,
            "hit_eos": bool(trace_df["is_eos"].any()),
        }
        # The summary row goes to disk too, so a resumed run can rebuild the
        # cross-model table from models it finished in an earlier session.
        (out / out_name("summary-row.json")).write_text(json.dumps(row, indent=2))
        return row
    finally:
        try:
            del model, tokenizer
        except NameError:
            pass
        free_gpu()


## 7. Run the sweep

> **The long cell.** Each model downloads (15-26 GiB), loads, then runs 100
> batched forward passes for the beam plus 100 for greedy. Download time
> dominates on a fast GPU.
>
> A failure in one model is recorded and the sweep continues - a model that is
> too big for this runtime, or that needs `trust_remote_code` you have not
> granted, must not cost you the other seven. Skips land in
> `results/skipped-models.csv` with the reason.
>
> **This cell is resumable.** If the runtime disconnects part-way, reconnect,
> re-run sections 0-6, then re-run this cell: every model whose
> `summary-row.json` is already on Drive is skipped, and the sweep continues
> from the first one that is not. The cross-model table is rebuilt from Drive
> rather than from this session, so earlier models still appear.


In [ ]:
# Re-running this cell after a disconnect picks up where it stopped: a model
# whose summary-row.json is already on Drive is not downloaded or run again.
# Set RERUN_COMPLETED = True to force the whole sweep from scratch.
RERUN_COMPLETED = False

failures = {}

for spec in MODELS:
    model_id = spec["id"]
    done = results_dir(model_id) / out_name("summary-row.json")

    if done.exists() and not RERUN_COMPLETED:
        print(f"\n=== {model_id}  - already done, skipping", flush=True)
        continue

    print(f"\n=== {model_id}  ({spec['bf16_GiB']} GiB)", flush=True)
    try:
        row = run_one(spec)
        print(f"  beam:   {row['beam_text']!r}", flush=True)
        print(f"  greedy: {row['greedy_text']!r}", flush=True)
    except Exception as exc:  # noqa: BLE001 - one model must not end the sweep
        failures[model_id] = f"{type(exc).__name__}: {exc}"
        print(f"  SKIPPED - {failures[model_id]}", flush=True)
    finally:
        # Always reclaim the disk, including after a failure part-way through a
        # download - otherwise a later model has nowhere to land.
        delete_weights(model_id)
        print(f"  weights removed, disk free {disk_free_gib():.0f} GiB", flush=True)

# Rebuild the cross-model table from what is on Drive, not from this session's
# results, so models finished in an earlier session are still represented.
order = {spec["id"]: i for i, spec in enumerate(MODELS)}
rows = [json.loads((results_dir(s["id"]) / out_name("summary-row.json")).read_text())
        for s in MODELS if (results_dir(s["id"]) / out_name("summary-row.json")).exists()]
summary_df = pd.DataFrame(sorted(rows, key=lambda r: order.get(r["model"], 99)))

print(f"\n{len(summary_df)} of {len(MODELS)} models have results on disk")
for model_id, why in failures.items():
    print(f"  failed this session: {model_id} - {why}")


## 8. Results

Same prompt, width, top-k and seed throughout, so every difference belongs to
the model.


In [ ]:
summary_df[[
    "model", "params_B", "bf16_GiB", "gpu_GiB", "blocks", "vocab",
    "download_s", "load_s", "search_s",
]]


### What the beam bought

`mean_pairwise_prefix` is the one to watch on an open-ended comic prompt: how
much of the returned sequences is *shared*. High means the beam collapsed onto a
single lineage and the four returned sequences are variations on one story; low
means genuinely distinct openings survived.

Compare `average_token_probability` rather than `total_log_probability` across
models - the total is a sum over generated tokens and so is length-sensitive.


In [ ]:
summary_df[[
    "model", "beam_log_p", "greedy_log_p", "gain_log_p",
    "beam_avg_token_p", "greedy_avg_token_p",
    "mean_pairwise_prefix", "distinct_first_tokens", "steps_run", "hit_eos",
]]


### The stories

What each architecture actually considered the most probable continuation.


In [ ]:
for row in summary_df.itertuples():
    print("=" * 100)
    print(f"{row.model}   [{row.family}]")
    print(f"  gain {row.gain_log_p:+.3f} logP  "
          f"({math.exp(row.gain_log_p):.2f}x more probable than greedy)")
    print(f"\n  BEAM  {row.beam_text}")
    print(f"\n  GREEDY {row.greedy_text}")
    print()


## 9. Export

`results/<model-slug>/` per model, plus a cross-model summary and, if anything
was skipped, the reasons. These are already on Drive; the zip is for the trip
back to the repo, where the folder names match `paths.results_dir()` exactly and
so drop straight into `results/`.


In [ ]:
summary_path = RESULTS / out_name("beam-search-teddy-bear.csv")
summary_df.to_csv(summary_path, index=False)

if failures:
    pd.DataFrame(
        [{"model": k, "reason": v} for k, v in failures.items()]
    ).to_csv(RESULTS / out_name("skipped-models.csv"), index=False)

archive = shutil.make_archive("/content/beam-search-teddy-bear-results", "zip", RESULTS)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)")

for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RESULTS)}")


In [ ]:
# The per-model results are already on Drive (see the Drive cell); this pulls
# the whole set down as one archive to unzip into the repo's results/ folder.
from google.colab import files
files.download(archive)


## Findings

Write these up in `docs/beam-search-teddy-bear.md` in the repo - a result that
lives only in a Colab session that will be recycled is not finished.

Each model's `run-metadata.json` already records what is needed to reproduce it:
model id, dtype, device, GPU, seed and the full `CONFIG`. Add the resolved
revision per model if you need provenance stronger than the id.
